# Etapa 3: Modelagem Preditiva

Comparacao de **Logistic Regression** (interpretavel) vs **Random Forest** (performance).
Validacao estratificada, tuning via RandomizedSearchCV e avaliacao final no holdout.

**SLAs:** Recall classe 1 >= 70%, Macro F1 >= 70%, gap treino/teste <= 10pp.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Backend nao-interativo para nbconvert
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score, recall_score, precision_score
)
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
import joblib

RANDOM_STATE = 42
REPO_ROOT = Path('..').resolve()
FIGURES_DIR = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregamento dos Dados Processados

In [2]:
df = pd.read_csv(REPO_ROOT / 'data' / 'processed' / 'telco_churn_features.csv')
print('Shape:', df.shape)

X = df.drop(columns=['Churn'])
y = df['Churn']

print(f'Features: {X.shape[1]}')
print(f'Target balance: {y.value_counts().to_dict()}')
print(f'Ratio: {y.value_counts()[0] / y.value_counts()[1]:.2f}:1')

Shape: (7043, 24)
Features: 23
Target balance: {0: 5174, 1: 1869}
Ratio: 2.77:1


## 2. Split Estratificado (Holdout 80/20)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Treino: {X_train.shape[0]} registros ({y_train.mean()*100:.1f}% churn)')
print(f'Teste:  {X_test.shape[0]} registros ({y_test.mean()*100:.1f}% churn)')

Treino: 5634 registros (26.5% churn)
Teste:  1409 registros (26.5% churn)


## 3. Definicao dos Pipelines

- **Logistic Regression:** `StandardScaler` embutido no Pipeline (evita data leakage).
- **Random Forest:** Sem scaler (arvores sao invariantes a escala).

In [4]:
# Cross-validation estratificado: 5 folds
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Pipeline 1: Logistic Regression (com scaler)
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ))
])

# Pipeline 2: Random Forest (sem scaler)
pipeline_rf = Pipeline([
    ('model', RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])

## 4. Tuning via RandomizedSearchCV

Busca aleatoria pragmatica — cobre o espaco de hiperparametros
sem o custo computacional de um GridSearch exaustivo.

In [5]:
# Hiperparametros para Logistic Regression
param_dist_lr = {
    'model__C': [0.01, 0.1, 0.5, 1, 5, 10],
    'model__solver': ['lbfgs', 'liblinear']
}

search_lr = RandomizedSearchCV(
    pipeline_lr,
    param_distributions=param_dist_lr,
    n_iter=12,  # 6 valores de C x 2 solvers = 12 combinacoes (cobre 100%)
    scoring='f1_macro',
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)
search_lr.fit(X_train, y_train)

print('Logistic Regression — Melhores hiperparametros:')
print(search_lr.best_params_)
print(f'Macro F1 (CV): {search_lr.best_score_:.4f}')

Logistic Regression — Melhores hiperparametros:
{'model__solver': 'lbfgs', 'model__C': 0.01}
Macro F1 (CV): 0.7209


In [6]:
# Hiperparametros para Random Forest
param_dist_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

search_rf = RandomizedSearchCV(
    pipeline_rf,
    param_distributions=param_dist_rf,
    n_iter=30,
    scoring='f1_macro',
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)
search_rf.fit(X_train, y_train)

print('Random Forest — Melhores hiperparametros:')
print(search_rf.best_params_)
print(f'Macro F1 (CV): {search_rf.best_score_:.4f}')

Random Forest — Melhores hiperparametros:
{'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_depth': 15}
Macro F1 (CV): 0.7405


## 5. Avaliacao no Holdout de Teste

In [7]:
def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te):
    """Avalia modelo no holdout e retorna dict de metricas."""
    y_pred_train = modelo.predict(X_tr)
    y_pred_test = modelo.predict(X_te)
    y_proba_test = modelo.predict_proba(X_te)[:, 1]

    # Metricas no teste
    recall_test = recall_score(y_te, y_pred_test)
    precision_test = precision_score(y_te, y_pred_test)
    f1_macro_test = f1_score(y_te, y_pred_test, average='macro')
    roc_auc_test = roc_auc_score(y_te, y_proba_test)

    # Gap de overfitting (F1 macro treino vs teste)
    f1_macro_train = f1_score(y_tr, y_pred_train, average='macro')
    gap = f1_macro_train - f1_macro_test

    metricas = {
        'Modelo': nome,
        'Recall (Churn)': recall_test,
        'Precision (Churn)': precision_test,
        'Macro F1 (Teste)': f1_macro_test,
        'Macro F1 (Treino)': f1_macro_train,
        'Gap (pp)': gap * 100,
        'ROC-AUC': roc_auc_test
    }

    print(f'\n=== {nome} ===')
    print(classification_report(y_te, y_pred_test, target_names=['No', 'Yes']))
    return metricas

metricas_lr = avaliar_modelo('Logistic Regression', search_lr.best_estimator_,
                              X_train, y_train, X_test, y_test)
metricas_rf = avaliar_modelo('Random Forest', search_rf.best_estimator_,
                              X_train, y_train, X_test, y_test)


=== Logistic Regression ===
              precision    recall  f1-score   support

          No       0.90      0.73      0.80      1035
         Yes       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409




=== Random Forest ===
              precision    recall  f1-score   support

          No       0.87      0.82      0.84      1035
         Yes       0.57      0.65      0.60       374

    accuracy                           0.77      1409
   macro avg       0.72      0.73      0.72      1409
weighted avg       0.79      0.77      0.78      1409



In [8]:
# Tabela comparativa
df_comp = pd.DataFrame([metricas_lr, metricas_rf]).set_index('Modelo')
df_comp = df_comp.round(4)
print(df_comp.to_string())

                     Recall (Churn)  Precision (Churn)  Macro F1 (Teste)  Macro F1 (Treino)  Gap (pp)  ROC-AUC
Modelo                                                                                                        
Logistic Regression          0.7807             0.5069            0.7094             0.7241    1.4697   0.8384
Random Forest                0.6471             0.5654            0.7229             0.9031   18.0231   0.8363


## 6. Matriz de Confusao e Curva ROC

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (nome, modelo) in zip(axes, [
    ('Logistic Regression', search_lr.best_estimator_),
    ('Random Forest', search_rf.best_estimator_)
]):
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes']).plot(ax=ax, cmap='Blues')
    ax.set_title(f'{nome}')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'modelo_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em reports/figures/modelo_confusion_matrix.png')

Salvo em reports/figures/modelo_confusion_matrix.png


In [10]:
fig, ax = plt.subplots(figsize=(8, 6))

for nome, modelo in [
    ('Logistic Regression', search_lr.best_estimator_),
    ('Random Forest', search_rf.best_estimator_)
]:
    y_proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{nome} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curva ROC — Comparacao dos Modelos')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'modelo_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em reports/figures/modelo_roc_curve.png')

Salvo em reports/figures/modelo_roc_curve.png


## 7. Selecao do Modelo Campeao e Calibracao de Probabilidades

In [11]:
# Selecionar campeao considerando os 3 SLAs:
# 1) Recall classe 1 >= 0.70
# 2) Macro F1 >= 0.70
# 3) Gap treino/teste <= 10pp
def atende_sla(m):
    return (m['Recall (Churn)'] >= 0.70 and
            m['Macro F1 (Teste)'] >= 0.70 and
            m['Gap (pp)'] <= 10)

lr_ok = atende_sla(metricas_lr)
rf_ok = atende_sla(metricas_rf)

print(f'LR atende todos os SLAs? {lr_ok}')
print(f'RF atende todos os SLAs? {rf_ok}')

# Regra de decisao: priorizar quem atende SLAs; desempate por Macro F1
if lr_ok and not rf_ok:
    nome_campeao = 'Logistic Regression'
    modelo_campeao = search_lr.best_estimator_
    params_campeao = search_lr.best_params_
elif rf_ok and not lr_ok:
    nome_campeao = 'Random Forest'
    modelo_campeao = search_rf.best_estimator_
    params_campeao = search_rf.best_params_
elif lr_ok and rf_ok:
    # Ambos atendem: desempate pelo Macro F1
    if metricas_rf['Macro F1 (Teste)'] > metricas_lr['Macro F1 (Teste)']:
        nome_campeao = 'Random Forest'
        modelo_campeao = search_rf.best_estimator_
        params_campeao = search_rf.best_params_
    else:
        nome_campeao = 'Logistic Regression'
        modelo_campeao = search_lr.best_estimator_
        params_campeao = search_lr.best_params_
else:
    # Nenhum atende: escolher pelo menor gap (mais robusto)
    if metricas_lr['Gap (pp)'] <= metricas_rf['Gap (pp)']:
        nome_campeao = 'Logistic Regression'
        modelo_campeao = search_lr.best_estimator_
        params_campeao = search_lr.best_params_
    else:
        nome_campeao = 'Random Forest'
        modelo_campeao = search_rf.best_estimator_
        params_campeao = search_rf.best_params_

print(f'\nModelo campeao: {nome_campeao}')
print(f'Hiperparametros: {params_campeao}')

LR atende todos os SLAs? True
RF atende todos os SLAs? False

Modelo campeao: Logistic Regression
Hiperparametros: {'model__solver': 'lbfgs', 'model__C': 0.01}


In [12]:
# Curva de Calibracao — avalia confiabilidade do predict_proba
fig, ax = plt.subplots(figsize=(8, 6))

for nome, modelo in [
    ('Logistic Regression', search_lr.best_estimator_),
    ('Random Forest', search_rf.best_estimator_)
]:
    CalibrationDisplay.from_estimator(modelo, X_test, y_test, n_bins=10,
                                      name=nome, ax=ax)

ax.set_title('Curva de Calibracao — Confiabilidade das Probabilidades')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'modelo_calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em reports/figures/modelo_calibration_curve.png')

Salvo em reports/figures/modelo_calibration_curve.png


In [13]:
# Se o Random Forest for campeao, calibrar as probabilidades
# para melhorar a qualidade do Score de Risco (Etapa 4)
if nome_campeao == 'Random Forest':
    print('RF selecionado — aplicando CalibratedClassifierCV...')
    # Extrai o modelo interno do pipeline
    rf_interno = modelo_campeao.named_steps['model']
    calibrador = CalibratedClassifierCV(rf_interno, cv=cv_strategy, method='isotonic')
    calibrador.fit(X_train, y_train)

    # Reconstruir pipeline calibrado para producao
    modelo_final = Pipeline([
        ('model', calibrador)
    ])

    # Verificar que a calibracao nao degradou a performance
    y_pred_cal = modelo_final.predict(X_test)
    y_proba_cal = modelo_final.predict_proba(X_test)[:, 1]
    print(f'Macro F1 pos-calibracao: {f1_score(y_test, y_pred_cal, average="macro"):.4f}')
    print(f'ROC-AUC pos-calibracao: {roc_auc_score(y_test, y_proba_cal):.4f}')
else:
    # LR ja eh naturalmente bem calibrada
    modelo_final = modelo_campeao
    print('LR selecionada — probabilidades naturalmente bem calibradas.')

LR selecionada — probabilidades naturalmente bem calibradas.


## 8. Interpretabilidade — Feature Importance

In [14]:
feature_names = X.columns.tolist()

# Se RF, usar feature_importances_
if nome_campeao == 'Random Forest':
    importances = search_rf.best_estimator_.named_steps['model'].feature_importances_
    df_importances = pd.DataFrame({
        'Feature': feature_names, 'Importancia': importances
    }).sort_values('Importancia', ascending=True)
    titulo = 'Feature Importance — Random Forest'
else:
    coefs = search_lr.best_estimator_.named_steps['model'].coef_[0]
    df_importances = pd.DataFrame({
        'Feature': feature_names, 'Importancia': np.abs(coefs)
    }).sort_values('Importancia', ascending=True)
    titulo = 'Feature Importance (|coeficientes|) — Logistic Regression'

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(df_importances['Feature'], df_importances['Importancia'], color='steelblue')
ax.set_title(titulo)
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'modelo_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo em reports/figures/modelo_feature_importance.png')

Salvo em reports/figures/modelo_feature_importance.png


## 9. Exportar Modelo Campeao

In [15]:
models_dir = REPO_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'campeao.joblib'
joblib.dump(modelo_final, model_path)
print(f'Modelo salvo em: {model_path}')
print(f'Tipo: {type(modelo_final)}')

Modelo salvo em: C:\Users\berna\.antigravity\projects\ds-synapsee-challenge\ds-synapsee-challenge\models\campeao.joblib
Tipo: <class 'sklearn.pipeline.Pipeline'>


## 10. Validacao dos SLAs

In [16]:
y_pred_final = modelo_final.predict(X_test)
y_proba_final = modelo_final.predict_proba(X_test)[:, 1]

recall_final = recall_score(y_test, y_pred_final)
f1_macro_final = f1_score(y_test, y_pred_final, average='macro')
roc_auc_final = roc_auc_score(y_test, y_proba_final)

# Gap treino/teste
y_pred_train_final = modelo_final.predict(X_train)
f1_macro_train_final = f1_score(y_train, y_pred_train_final, average='macro')
gap_final = (f1_macro_train_final - f1_macro_final) * 100

print('=== VALIDACAO DOS SLAs ===')
print(f'Recall classe Churn:  {recall_final:.4f} (SLA >= 0.70) -> {"PASS" if recall_final >= 0.70 else "FAIL"}')
print(f'Macro F1-Score:       {f1_macro_final:.4f} (SLA >= 0.70) -> {"PASS" if f1_macro_final >= 0.70 else "FAIL"}')
print(f'Gap treino/teste:     {gap_final:.1f}pp (SLA <= 10pp) -> {"PASS" if gap_final <= 10 else "FAIL"}')
print(f'ROC-AUC:              {roc_auc_final:.4f}')

=== VALIDACAO DOS SLAs ===
Recall classe Churn:  0.7807 (SLA >= 0.70) -> PASS
Macro F1-Score:       0.7094 (SLA >= 0.70) -> PASS
Gap treino/teste:     1.5pp (SLA <= 10pp) -> PASS
ROC-AUC:              0.8384
